# PMS: cleaning `hotel_reservations_pms.xlsx` (Phase A)

Load, Assess and Clean of the single file. For each step: the problem, the decision with its reason, the rows before and after.

- **CHECK** cells show the numbers the next decision is based on. They do not change the data.
- **Phase A rule:** a value becomes NaN only if it is impossible on its own. Recovering it from the other file (CRM) happens in Phase B, after the merge.
- **Cleaning before the merge:** a step on a single file must not harm the merge, and must not delete information that the other file can recompute or check.
- The notebook lives in `Project/cleaning`, reads the raw data from `Project/data/raw` and writes the clean table to `Project/data/processed`.

In [1]:
import pandas as pd

## 1. Load
We read the `bookings` sheet: the other sheet, `info`, is only the column dictionary. The raw data stays untouched in `raw`; every change is made on the copy `pms`.

In [2]:
raw = pd.read_excel("../data/raw/hotel_reservations_pms.xlsx", sheet_name="bookings")
pms = raw.copy()
raw.shape

(24480, 20)

## 2. Assess on the raw data (no changes)

### Missing values
`meal` also contains `Undefined`. It is not in the `info` dictionary (BB, HB, FB, SC), but the original paper of the dataset (Antonio, de Almeida, Nunes, *Data in Brief* 22, 2019) defines it as "Undefined/SC – no meal package", i.e. the same as SC.

In [3]:
print("meal = 'Undefined':", (raw["meal"] == "Undefined").sum())
missing = raw.isna().sum()
missing[missing > 0]

meal = 'Undefined': 238


children    815
meal        490
dtype: int64

### Text inconsistencies
The same categories are written in different ways.

In [4]:
print("hotel, spellings before/after strip+title:", raw["hotel"].nunique(),
      raw["hotel"].str.strip().str.title().nunique())
print("deposit_type, spellings before/after strip+title:", raw["deposit_type"].nunique(),
      raw["deposit_type"].str.strip().str.title().nunique())
print("arrival_date_month, different spellings:", raw["arrival_date_month"].astype(str).nunique())
raw["is_canceled"].astype(str).value_counts()

hotel, spellings before/after strip+title: 10 2
deposit_type, spellings before/after strip+title: 5 3
arrival_date_month, different spellings: 47


is_canceled
0      14579
1       8679
no       786
yes      436
Name: count, dtype: int64

### adr stored as text, and duplicates
We count the `adr` values that cannot be read as a number, and how many of them contain no digit at all (placeholders such as "n.d." or "?"). Then we count identical rows and repeated booking codes.

In [5]:
adr_text = raw["adr"].astype(str)
adr_not_number = pd.to_numeric(raw["adr"], errors="coerce").isna()
print("adr not numeric:", adr_not_number.sum(),
      "| of which without digits:", (adr_not_number & ~adr_text.str.contains("[0-9]")).sum())
print("exact copies:", raw.duplicated().sum())
print("repeated booking_id:", raw["booking_id"].duplicated().sum())

adr not numeric: 608 | of which without digits: 122
exact copies: 150
repeated booking_id: 230


## 3. Clean

### 3.1 Inconsistencies: column types
The three columns with the wrong type contain dirty text, so they cannot be converted in one go: values would be lost. First we make the spelling uniform, then we convert.
- `arrival_date_month` becomes a number from 1 to 12.
- `is_canceled` becomes 0/1.
- `adr`: we remove the symbol and the word, the comma becomes a dot, then we convert to a number. Values without digits become missing: no price is invented.

In [6]:
months = {"january": 1, "february": 2, "march": 3, "april": 4, "may": 5, "june": 6,
          "july": 7, "august": 8, "september": 9, "october": 10, "november": 11, "december": 12,
          "jan": 1, "feb": 2, "mar": 3, "apr": 4, "jun": 6, "jul": 7, "aug": 8,
          "sep": 9, "oct": 10, "nov": 11, "dec": 12}
for n in range(1, 13):
    months[str(n)] = n
pms["arrival_date_month"] = pms["arrival_date_month"].astype(str).str.strip().str.lower().map(months)

pms["is_canceled"] = pms["is_canceled"].astype(str).str.strip().str.lower().map(
    {"0": 0, "1": 1, "no": 0, "yes": 1})

adr = pms["adr"].astype(str).str.replace("€", "", regex=False).str.replace("EUR", "", regex=False)
adr = adr.str.replace(",", ".", regex=False).str.strip()
pms["adr"] = pd.to_numeric(adr, errors="coerce")

print("months not recognised:", pms["arrival_date_month"].isna().sum())
print("is_canceled not recognised:", pms["is_canceled"].isna().sum())
print("adr missing after conversion:", pms["adr"].isna().sum())

months not recognised: 0
is_canceled not recognised: 0
adr missing after conversion: 122


### 3.2 Inconsistencies: text of hotel, deposit_type and meal
- `hotel` and `deposit_type`: we strip the spaces and bring everything to the same format (e.g. "city hotel" → "City Hotel").
- `meal`: "Undefined" and "SC" are two ways of writing "no meal" (source: the original paper, see Assess). We recode it to SC. First we create the column `meal_recoded`, which records which rows were changed.

In [7]:
pms["hotel"] = pms["hotel"].str.strip().str.title()
pms["deposit_type"] = pms["deposit_type"].str.strip().str.title()

pms["meal_recoded"] = pms["meal"] == "Undefined"
pms["meal"] = pms["meal"].replace("Undefined", "SC")
print("meal recoded to SC:", pms["meal_recoded"].sum())

meal recoded to SC: 238


### CRM, read only for checks
The CRM is not cleaned or merged here: that is Phase B. Here it is a second source to understand the cause of the PMS problems. We only drop rows without a code and repeated codes, and convert `total_price` (decimal comma) to a number.

In [8]:
crm = pd.read_csv("../data/raw/hotel_bookings_crm.csv", sep=";")
crm = crm.dropna(subset=["BookingRef"]).drop_duplicates(subset="BookingRef", keep="last")
crm = crm.set_index("BookingRef")
crm_price = pd.to_numeric(crm["total_price"].astype(str).str.replace(",", ".", regex=False),
                          errors="coerce")
crm_adults = crm["n_adults"]
crm.shape

(24500, 17)

### 3.3 Duplicates
First the exact copies: keeping one is enough.

In [9]:
rows_before = len(pms)
pms = pms.drop_duplicates()
print("exact copies removed, rows:", rows_before, "->", len(pms))

exact copies removed, rows: 24480 -> 24330


**CHECK of the versions.** Some bookings still have the same code but different rows. We compare the old and the new version (ordered by `record_updated_at`) column by column. The table shows only the columns that change.

In [10]:
pms = pms.sort_values("record_updated_at")
repeated = pms[pms.duplicated(subset="booking_id", keep=False)]
v_old = repeated.drop_duplicates(subset="booking_id", keep="first")
v_old = v_old.sort_values("booking_id").reset_index(drop=True)
v_new = repeated.drop_duplicates(subset="booking_id", keep="last")
v_new = v_new.sort_values("booking_id").reset_index(drop=True)
print("bookings with two versions:", len(v_new))

differences = {}
for col in pms.columns:
    both_empty = v_old[col].isna() & v_new[col].isna()
    differ = ((v_old[col] != v_new[col]) & ~both_empty).sum()
    only_old = (v_new[col].isna() & v_old[col].notna()).sum()
    if differ > 0:
        differences[col] = {"pairs that differ": differ, "value only in the old one": only_old}
pd.DataFrame(differences).T

bookings with two versions: 80


,pairs that differ,value only in the old one
children,1,1
meal,2,2
booking_changes,12,0
adr,80,0
record_updated_at,80,0


In [11]:
adr_ratio = v_new["adr"] / v_old["adr"]
days = (v_new["record_updated_at"] - v_old["record_updated_at"]).dt.days
print("adr new / old: min", round(adr_ratio.min(), 3), "max", round(adr_ratio.max(), 3))
print("days between versions: min", days.min(), "max", days.max())
print("booking_changes new - old:",
      (v_new["booking_changes"] - v_old["booking_changes"]).value_counts().to_dict())
nights_v = v_new["stays_in_weekend_nights"] + v_new["stays_in_week_nights"]
crm_adr_v = v_new["booking_id"].map(crm_price) / nights_v
print("versions checkable on the CRM:", crm_adr_v.notna().sum(),
      "| CRM = new adr:", ((crm_adr_v / v_new["adr"] - 1).abs() < 0.01).sum(),
      "| CRM = old adr:", ((crm_adr_v / v_old["adr"] - 1).abs() < 0.01).sum())

adr new / old: min 1.118 max 1.33
days between versions: min 2 max 39
booking_changes new - old: {0: 68, 1: 12}
versions checkable on the CRM: 77 | CRM = new adr: 76 | CRM = old adr: 0


**Decision:** the most recent version wins, and the CRM confirms it. In 3 cases, however, the new version has a gap that the old one does not (1 in `children`, 2 in `meal`). With `combine_first` we keep the new one and fill only its gaps from the old one: this is the "most complete" rule of the golden record.

It is an assumption. In those 3 cases `booking_changes` is 0 and, among the known values, `meal` and `children` never change from one version to the other.

In [12]:
cols = pms.columns
rows_before = len(pms)
latest = pms.drop_duplicates(subset="booking_id", keep="last").set_index("booking_id")
older = pms[pms.duplicated(subset="booking_id", keep="last")].set_index("booking_id")
pms = latest.combine_first(older).reset_index()
pms = pms[cols]
print("versions merged, rows:", rows_before, "->", len(pms))

versions merged, rows: 24330 -> 24250


### 3.4 Impossible values
**CHECK:** how many there are and what the CRM says about their cause. `crm_adr = total_price / nights` is the formula the merge will use, so first we check that it works on the normal rows.

In [13]:
nights = pms["stays_in_weekend_nights"] + pms["stays_in_week_nights"]
price = pms["booking_id"].map(crm_price)
crm_adr = price / nights
checkable = price.notna() & (nights > 0)
normal = checkable & (pms["adr"] > 0) & (pms["adr"] <= 1000)
print("formula total_price / nights = adr on normal rows:",
      round(((crm_adr / pms["adr"] - 1).abs() < 0.01)[normal].mean(), 3))

negative = pms["adr"] < 0
print("negative adr:", negative.sum(), "| checkable:", (negative & checkable).sum(),
      "| CRM = same value but positive:",
      ((crm_adr / -pms["adr"] - 1).abs() < 0.01)[negative & checkable].sum())

high = pms["adr"] > 1000
print("adr above 1000:", high.sum(), "| highest below 1000:", pms["adr"][pms["adr"] <= 1000].max(),
      "| lowest above 1000:", pms["adr"][high].min())
print("   checkable:", (high & checkable).sum(),
      "| CRM = adr / 100:", ((pms["adr"] / 100 / crm_adr - 1).abs() < 0.01)[high & checkable].sum(),
      "| CRM = adr:", ((crm_adr / pms["adr"] - 1).abs() < 0.01)[high & checkable].sum())

zero = pms["adr"] == 0
print("adr = 0:", zero.sum(), "| with CRM price:", (zero & price.notna()).sum(),
      "| CRM = 0:", (zero & (price == 0)).sum())

formula total_price / nights = adr on normal rows: 0.988
negative adr: 40 | checkable: 39 | CRM = same value but positive: 39
adr above 1000: 31 | highest below 1000: 387.0 | lowest above 1000: 3511.0
   checkable: 27 | CRM = adr / 100: 26 | CRM = adr: 1
adr = 0: 377 | with CRM price: 362 | CRM = 0: 362


In [14]:
minus_one = pms["stays_in_week_nights"] < 0
week_rebuilt = (price / pms["adr"]).round() - pms["stays_in_weekend_nights"]
print("negative stays_in_week_nights:", minus_one.sum(),
      "| values:", pms["stays_in_week_nights"][minus_one].value_counts().to_dict())
print("   true value rebuilt from the CRM:",
      week_rebuilt[minus_one & (pms["adr"] > 0)].value_counts().sort_index().to_dict())

print("bookings with 0 nights:", (nights == 0).sum(),
      "| of which only because of weekend + (-1):", ((nights == 0) & minus_one).sum())

guests = pms["adults"] + pms["children"] + pms["babies"]
print("no guests:", (guests == 0).sum(),
      "| 0 adults, 0 babies and children missing:",
      ((pms["adults"] == 0) & (pms["babies"] == 0) & pms["children"].isna()).sum())

many = pms["adults"] > 10
print("more than 10 adults:", many.sum())
pms["booking_id"].map(crm_adults)[many].value_counts(dropna=False)

negative stays_in_week_nights: 35 | values: {-1: 35}
   true value rebuilt from the CRM: {1.0: 10, 2.0: 10, 3.0: 4, 4.0: 3, 5.0: 2, 6.0: 2}
bookings with 0 nights: 159 | of which only because of weekend + (-1): 12
no guests: 31 | 0 adults, 0 babies and children missing: 7
more than 10 adults: 23


booking_id
2.0     15
1.0      4
NaN      2
26.0     1
55.0     1
Name: count, dtype: int64

**CHECK of consistency** between columns that depend on each other.

In [15]:
arrival = pd.to_datetime(pms["arrival_date_year"].astype(str) + "-"
                         + pms["arrival_date_month"].astype(str) + "-"
                         + pms["arrival_date_day_of_month"].astype(str), errors="coerce")
print("invalid arrival dates:", arrival.isna().sum(), "| from", arrival.min().date(), "to", arrival.max().date())
counters = ["adults", "children", "babies", "booking_changes", "required_car_parking_spaces",
            "total_of_special_requests", "stays_in_weekend_nights"]
print("negative values in the counters:", (pms[counters] < 0).sum().sum())
print("updated more than 30 days after departure:",
      ((pms["record_updated_at"] - arrival).dt.days > nights + 30).sum())
print("more parking spaces than guests:", (pms["required_car_parking_spaces"] > guests).sum(),
      "| cancelled with parking spaces:",
      ((pms["is_canceled"] == 1) & (pms["required_car_parking_spaces"] > 0)).sum())

invalid arrival dates: 0 | from 2015-07-01 to 2017-08-31
negative values in the counters: 0
updated more than 30 days after departure: 0
more parking spaces than guests: 2 | cancelled with parking spaces: 0


**Weekend nights.** We count how many nights of a given pair of days fall in the stay, knowing the arrival weekday (0 = Monday … 6 = Sunday) and the total nights. Every full week contains 2 of them. In the remaining days, a day g falls in the stay if its distance from the arrival is smaller than the remaining days.

In [16]:
weekday = arrival.dt.dayofweek
weeks = nights // 7
rest = nights % 7
sat_sun = weeks * 2 + ((5 - weekday) % 7 < rest).astype(int) + ((6 - weekday) % 7 < rest).astype(int)
sun_mon = weeks * 2 + ((6 - weekday) % 7 < rest).astype(int) + ((7 - weekday) % 7 < rest).astype(int)
checkable_nights = ~minus_one
print("weekend nights = Saturday and Sunday nights:",
      round((sat_sun == pms["stays_in_weekend_nights"])[checkable_nights].mean(), 3))
print("weekend nights = Sunday and Monday nights:",
      round((sun_mon == pms["stays_in_weekend_nights"])[checkable_nights].mean(), 3))

weekend nights = Saturday and Sunday nights: 0.498
weekend nights = Sunday and Monday nights: 1.0


Result: the rule is consistent, but shifted by one day (Sunday and Monday). Nothing is corrected. In Phase B it will be used to check the rebuilt nights.

**Decisions.** Before changing a value we create the column that marks it. This way, in Phase B and in the before/after charts, the original missing values can be told apart from the ones created here.

**adr negative or above 1,000 euro.** A price per night cannot be negative. Above 387 euro there is a jump up to 3,511, so the threshold falls in an empty gap and is not arbitrary. These values become NaN, and in Phase B they are recomputed from `total_price / nights` of the CRM. Rows without a CRM match will stay NaN: it is an intended loss, because the cause is not proven row by row. BK323523 (5,400 euro for 1 night) is confirmed by the CRM: in Phase B it will go back to 5,400 and be marked as an outlier.

**adr = 0.** Kept, because the CRM also says 0 (free stays, group or agency packages, day use). Marked with `adr_zero`: in the price analysis these rows must be excluded, otherwise they lower the mean.

In [17]:
pms["adr_invalid"] = (pms["adr"] < 0) | (pms["adr"] > 1000)
pms["adr"] = pms["adr"].where(pms["adr"].between(0, 1000))
print("adr set to NaN:", pms["adr_invalid"].sum())

pms["adr_zero"] = pms["adr"] == 0
print("adr = 0 marked:", pms["adr_zero"].sum())

adr set to NaN: 71
adr = 0 marked: 377


**stays_in_week_nights = -1.** It is a code for "unknown", not a wrong sign: the true value goes from 1 to 6. It becomes NaN. In Phase B it is rebuilt first from the CRM dates (check-outs only), then from `total_price / adr` minus the weekend nights.

**0 nights.** Kept: they are one-day stays, almost all checked out in the CRM. The flag is computed after removing the -1, so rows with weekend 1 and week -1 are not counted.

In [18]:
pms["week_nights_missing"] = pms["stays_in_week_nights"] < 0
pms["stays_in_week_nights"] = pms["stays_in_week_nights"].where(pms["stays_in_week_nights"] >= 0)
print("week nights set to NaN:", pms["week_nights_missing"].sum())

pms["zero_nights"] = (pms["stays_in_weekend_nights"] + pms["stays_in_week_nights"]) == 0
print("0 nights marked:", pms["zero_nights"].sum())

week nights set to NaN: 35
0 nights marked: 147


**No guests.** Kept and marked. The true number of guests is unknown, and the CRM does not help: it also says 0, but it probably copies the same source data, so it is not an independent confirmation. The flag is computed before filling `children`: rows with missing `children` are not marked, because their "zero children" would only be an estimate.

**More than 10 adults.** No change in this phase. The value is not impossible on its own: real groups exist, and some are confirmed by the CRM. An adr above 0 is only a hint. The CRM decides in Phase B, row by row.

In [19]:
pms["no_guests"] = (pms["adults"] + pms["children"] + pms["babies"]) == 0
print("no guests marked:", pms["no_guests"].sum())

no guests marked: 31


### 3.5 Missing values
Handled last, because the impossible-values step creates new ones.

In [20]:
missing = pms.isna().sum()
missing[missing > 0]

stays_in_week_nights     35
children                809
meal                    483
adr                     191
dtype: int64

**CHECK:** where the missing values of `children` and `meal` are, and what the distribution of `children` looks like.

In [21]:
children_empty = pms["children"].isna()
share_month = children_empty[pms["hotel"] == "City Hotel"].groupby(pms["arrival_date_month"]).mean()
share_year = children_empty[pms["hotel"] == "City Hotel"].groupby(pms["arrival_date_year"]).mean()
print("children missing in the City Hotel, by month: min", round(share_month.min(), 3),
      "max", round(share_month.max(), 3),
      "| by year: min", round(share_year.min(), 3), "max", round(share_year.max(), 3))
print("known children: mean", round(pms["children"].mean(), 2), "| median", pms["children"].median(),
      "| mode", pms["children"].mode()[0], "| share of zeros",
      round((pms["children"] == 0).sum() / pms["children"].notna().sum(), 3))
pd.DataFrame({"children missing": children_empty.groupby(pms["hotel"]).mean(),
              "meal missing": pms["meal"].isna().groupby(pms["hotel"]).mean()}).round(3)

children missing in the City Hotel, by month: min 0.035 max 0.062 | by year: min 0.047 max 0.052
known children: mean 0.1 | median 0.0 | mode 0.0 | share of zeros 0.93


,children missing,meal missing
hotel,,
City Hotel,0.05,0.018
Resort Hotel,0.00,0.023


**Decisions.**

**meal.** The truly missing values stay empty. We do not know which meal was booked, and the mode (BB) would assign a wrong meal to part of the rows.

**children.** The missing values are only in the City Hotel and are spread evenly across months and years. They cannot be derived from other columns, and dropping the rows would only remove City Hotel bookings. We fill them with 0: the mean (0.1) would give a number of children that does not exist, while mode and median agree and are both 0. It is an **ESTIMATE**: before filling we create the column `children_missing`, which records which rows were filled.

**adr.** The missing values, both the original ones and those created in 3.4, are derived after the merge.

In [22]:
pms["children_missing"] = pms["children"].isna()
pms["children"] = pms["children"].fillna(0)
print("children filled with 0:", pms["children_missing"].sum())

children filled with 0: 809


## End of Phase A
Final order by booking code, so the output is stable. Below: the final shape and how many rows each flag marks.

In [23]:
pms = pms.sort_values("booking_id").reset_index(drop=True)
print("rows, columns:", pms.shape)
flags = ["meal_recoded", "adr_invalid", "adr_zero", "week_nights_missing",
         "zero_nights", "no_guests", "children_missing"]
pms[flags].sum()

rows, columns: (24250, 27)


meal_recoded           236
adr_invalid             71
adr_zero               377
week_nights_missing     35
zero_nights            147
no_guests               31
children_missing       809
dtype: int64

## Export
The clean table is saved in `data/processed/`, while the raw files in `data/raw/` stay untouched. Phase B reads it back with `pd.read_csv`.

In [24]:
pms.to_csv("../data/processed/pms_clean.csv", index=False)
print("saved ../data/processed/pms_clean.csv:", pms.shape)

saved ../data/processed/pms_clean.csv: (24250, 27)


## Notes for Phase B (merge with the CRM)

Before using the CRM to fill values, we check that the shared columns agree:
- `hotel` / `property`;
- arrival date / `checkin_date`;
- `adults` / `n_adults`;
- `is_canceled` / `reservation_status`;
- nights / difference between the CRM dates (check-outs only);
- `total_price` / adr times nights;
- the Sunday-Monday rule on the rebuilt nights;
- arrival minus `lead_time` not later than `record_updated_at`;
- `is_repeated_guest` / previous bookings.

To do:
- **adr:**
  - recompute from `total_price / nights`;
  - rows with missing adr and 0 nights become 0, because all 0-night rows with a known adr have adr 0;
  - BK323523 must be marked as an outlier;
  - in about 278 rows the CRM is higher than the PMS (by 15% to 58%): there the derivation must be checked.
- **week_nights:** first the CRM dates for check-outs, then `total_price / adr` minus the weekend, only if the result is a whole number. The rebuilt values are an ESTIMATE.
- **Adults above 10:** compare with `n_adults` of the CRM.
- **Conflicts already seen:**
  - about 56 rows with different adults in the two files;
  - about 46 arrival dates shifted by a few days;
  - CRM dates in mixed formats.